# Moises without the Noises — Song Processing Notebook

Runs on Colab's free GPU to produce everything the app needs for one song:
- **6-stem separation** via Demucs `htdemucs_6s` (vocals, drums, bass, other, guitar, piano)
- **Lyric transcription** via OpenAI Whisper (word-level timestamps)
- **BPM + beat detection** via librosa
- **Key detection** via librosa (Krumhansl-Schmuckler)
- **Note detection** via librosa pyin (vocals, bass)

Output is a zip you drop directly into `backend/data/`. The app picks it up automatically.

---

### Before running
**Runtime → Change runtime type → T4 GPU** (free tier is fine)

### Mobile data warning
The first run downloads ~3.5 GB of model weights (Demucs + Whisper medium).
These are cached in `/root/.cache/` for the duration of the Colab session,
but **are lost when the session ends** unless you mount Google Drive (Cell 2 shows how).
Subsequent cells in the same session cost ~0 extra data.

### Known limitations
- Note detection uses `librosa.pyin` — a **monophonic** pitch tracker. Reliable on
  isolated bass and vocal stems. The `other` stem is often polyphonic and results there
  are best-effort. Guitar and piano stems don't get note detection (chords).
- Whisper transcription is English-optimized by default. Change `WHISPER_LANGUAGE`
  below for other languages.
- `htdemucs_6s` is slightly slower than `htdemucs` (4-stem). On a T4, expect
  ~30–90 seconds per song depending on length.

## Cell 1 — Install dependencies

**Data cost: ~3.5 GB on first run** (Demucs weights ~2.3 GB + Whisper medium ~1.5 GB).
Subsequent runs in the same Colab session: ~0 (cached).
If you mount Google Drive in Cell 2, weights persist across sessions.

In [1]:
!pip install -q demucs openai-whisper librosa soundfile

# Verify GPU is available
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. Processing will be significantly slower.')
    print('Go to Runtime → Change runtime type → GPU and re-run.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2 — (Optional) Mount Google Drive to cache model weights

Skip this if you're on an unmetered connection and don't mind re-downloading
weights each session (~3.5 GB). If you're on mobile data, run this once to
cache weights to your Drive and avoid the download on future sessions.

In [3]:
CACHE_WEIGHTS_TO_DRIVE = False  # Set to True to enable Drive caching

if CACHE_WEIGHTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    DRIVE_CACHE = '/content/drive/MyDrive/mwtn_model_cache'
    os.makedirs(DRIVE_CACHE, exist_ok=True)

    # Point both Demucs and Whisper at the Drive cache directory
    # so weights downloaded once persist across sessions.
    os.environ['TORCH_HOME'] = DRIVE_CACHE
    os.environ['XDG_CACHE_HOME'] = DRIVE_CACHE
    print(f'Model weights will be cached to: {DRIVE_CACHE}')
    print('First run still downloads weights; subsequent sessions will load from Drive.')
else:
    print('Drive caching disabled. Weights download fresh each session.')

Drive caching disabled. Weights download fresh each session.


## Cell 3 — Configuration

Set your preferences here before running the rest.

In [4]:
# --- Model selection ---------------------------------------------------------

# htdemucs_6s: 6 stems (vocals, drums, bass, other, guitar, piano) — recommended
# htdemucs:    4 stems (vocals, drums, bass, other) — faster, use if 6s is too slow
DEMUCS_MODEL = 'htdemucs_6s'

# Whisper model size. Tradeoffs:
#   tiny   — fastest, least accurate (~75MB)
#   base   — fast, decent accuracy (~150MB)
#   small  — good balance (~500MB)
#   medium — best open-source accuracy (~1.5GB) — recommended on GPU
#   large  — best accuracy, slow even on GPU (~3GB)
WHISPER_MODEL = 'medium'

# Language for Whisper transcription.
# None = auto-detect. Explicit is faster and more accurate when you know it.
# Options: 'english', 'spanish', 'portuguese', 'french', 'italian', etc.
WHISPER_LANGUAGE = None

print(f'Demucs model:  {DEMUCS_MODEL}')
print(f'Whisper model: {WHISPER_MODEL}')
print(f'Language:      {WHISPER_LANGUAGE or "auto-detect"}')

Demucs model:  htdemucs_6s
Whisper model: medium
Language:      auto-detect


## Cell 4 — Load your song

**Pick ONE of the three methods below** and set `UPLOAD_METHOD` accordingly.
Comment out the other two blocks. Any common audio format works: mp3, wav, flac, m4a, ogg.

| Method | When to use |
|---|---|
| `'drive'` | **Recommended.** Upload the file to Google Drive first, then paste its path. Most reliable. |
| `'url'` | Song is publicly accessible via a direct download URL (e.g. a Dropbox/GDrive share link). |
| `'widget'` | Last resort. The Colab file-picker widget — works but can time out if you're slow to click. |

**Data cost:** just the song file (~3–10 MB for mp3).

In [5]:
from pathlib import Path

# ── Pick your method ──────────────────────────────────────────────────────────
UPLOAD_METHOD = 'drive'  # 'drive' | 'url' | 'widget'

# Used by 'drive' method: path relative to /content/drive/MyDrive/
# Example: 'Music/bohemian_rhapsody.mp3'
DRIVE_FILE_PATH = 'Music/Dunsin-Oyekan-You-Remain-Thesame-1.mp3'

# Used by 'url' method: must be a DIRECT download link, not a webpage.
# Dropbox: change ?dl=0 to ?dl=1 at the end of the share URL.
# Google Drive: use https://drive.google.com/uc?export=download&id=FILE_ID
DIRECT_URL = ''
# ─────────────────────────────────────────────────────────────────────────────

input_filename = None

if UPLOAD_METHOD == 'drive':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    src = Path('/content/drive/MyDrive') / DRIVE_FILE_PATH
    if not src.exists():
        raise FileNotFoundError(
            f'File not found in Drive: {src}\n'
            f'Make sure the file is uploaded to your Google Drive at that path.'
        )
    # Copy to Colab local storage so Demucs can write next to it freely.
    import shutil
    dest = Path('/content') / src.name
    shutil.copy(src, dest)
    input_filename = str(dest)
    print(f'Loaded from Drive: {src.name}')

elif UPLOAD_METHOD == 'url':
    import urllib.request
    if not DIRECT_URL:
        raise ValueError('Set DIRECT_URL to a direct download link.')
    # Derive filename from URL; fall back to 'song.mp3' if URL has no extension.
    url_filename = Path(DIRECT_URL.split('?')[0]).name or 'song.mp3'
    dest = Path('/content') / url_filename
    print(f'Downloading {url_filename}...')
    urllib.request.urlretrieve(DIRECT_URL, dest)
    input_filename = str(dest)
    print(f'Downloaded: {dest} ({dest.stat().st_size / 1e6:.1f} MB)')

elif UPLOAD_METHOD == 'widget':
    # The original file-picker widget. Works, but blocks until you select
    # a file — if you take too long or the popup is blocked, it raises
    # KeyboardInterrupt. Click the 'Choose Files' button immediately.
    from google.colab import files
    print('A file picker will appear below. Select your file immediately.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file selected.')
    input_filename = list(uploaded.keys())[0]
    print(f'Uploaded: {input_filename}')

else:
    raise ValueError(f'Unknown UPLOAD_METHOD: {UPLOAD_METHOD!r}. Use drive, url, or widget.')

input_path = Path(input_filename)
print(f'Song file: {input_path} ({input_path.stat().st_size / 1e6:.1f} MB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded from Drive: Dunsin-Oyekan-You-Remain-Thesame-1.mp3
Song file: /content/Dunsin-Oyekan-You-Remain-Thesame-1.mp3 (13.7 MB)


## Cell 5 — Stem separation (Demucs)

This is the GPU-heavy step. Typical time on T4:
- 3-min song → ~30–45 seconds
- 5-min song → ~60–90 seconds

Progress bar will appear below.

In [6]:
import subprocess
import sys
from pathlib import Path

# input_path is already set by Cell 4 — don't re-derive it here.
# Sanitise the stem: lowercase, spaces to underscores, strip non-alphanum.
import re
raw_stem = input_path.stem
song_id = re.sub(r'[^\w]', '_', raw_stem).strip('_') or 'song'
demucs_out = Path('_demucs_raw')

print(f'Song ID: {song_id}')
print(f'Running Demucs ({DEMUCS_MODEL})...')

result = subprocess.run(
    [sys.executable, '-m', 'demucs', '-n', DEMUCS_MODEL, '-o', str(demucs_out), str(input_path)],
    capture_output=False,  # Let tqdm progress bar render live
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(f'Demucs failed. Check output above.')

stems_source_dir = demucs_out / DEMUCS_MODEL / input_path.stem
stem_wavs = list(stems_source_dir.glob('*.wav'))
print(f'\nSeparation complete. Stems: {[w.stem for w in stem_wavs]}')

Song ID: Dunsin_Oyekan_You_Remain_Thesame_1
Running Demucs (htdemucs_6s)...

Separation complete. Stems: ['piano', 'guitar', 'bass', 'drums', 'vocals', 'other']


## Cell 6 — Lyric transcription (Whisper)

Transcribes the **vocals stem** (not the full mix) for cleaner results.
Returns word-level timestamps the frontend uses to sync lyrics to playback.

This cell is skipped automatically if no vocals stem is available.

In [7]:
import whisper
import json

vocals_path = stems_source_dir / 'vocals.wav'

lyrics_result = None

if not vocals_path.exists():
    print('No vocals stem found — skipping transcription.')
else:
    print(f'Loading Whisper ({WHISPER_MODEL})...')
    whisper_model = whisper.load_model(WHISPER_MODEL)

    print('Transcribing vocals stem...')
    transcribe_kwargs = {
        'word_timestamps': True,  # needed for word-level sync in the frontend
        'verbose': False,
    }
    if WHISPER_LANGUAGE:
        transcribe_kwargs['language'] = WHISPER_LANGUAGE

    result = whisper_model.transcribe(str(vocals_path), **transcribe_kwargs)

    # Flatten to word-level segments for the frontend.
    # Each word: {start, end, word}
    words = []
    for seg in result.get('segments', []):
        for w in seg.get('words', []):
            words.append({
                'start': round(w['start'], 3),
                'end': round(w['end'], 3),
                'word': w['word'].strip(),
            })

    # Also keep full-segment text for display
    segments = [{
        'start': round(s['start'], 3),
        'end': round(s['end'], 3),
        'text': s['text'].strip(),
    } for s in result.get('segments', [])]

    lyrics_result = {
        'language': result.get('language', 'unknown'),
        'words': words,
        'segments': segments,
    }

    print(f'Language detected: {lyrics_result["language"]}')
    print(f'Words transcribed: {len(words)}')
    if segments:
        print(f'First line: {segments[0]["text"]}')

    # Free GPU memory — Whisper is no longer needed
    del whisper_model
    torch.cuda.empty_cache()

Loading Whisper (medium)...


100%|██████████████████████████████████████| 1.42G/1.42G [00:12<00:00, 120MiB/s]


Transcribing vocals stem...
Detected language: Yoruba


100%|██████████| 56907/56907 [01:06<00:00, 849.45frames/s] 

Language detected: yo
Words transcribed: 763
First line: A


## Cell 7 — BPM & beat detection

In [8]:
import librosa
import numpy as np

# Use drums stem for beat detection if available — most reliable signal.
# Fall back to 'other' or whatever is available.
beat_stem_priority = ['drums', 'other', 'bass', 'vocals']
beat_stem_path = None
for stem_name in beat_stem_priority:
    candidate = stems_source_dir / f'{stem_name}.wav'
    if candidate.exists():
        beat_stem_path = candidate
        print(f'Using {stem_name} stem for BPM detection')
        break

beats_result = None

if beat_stem_path:
    y, sr = librosa.load(str(beat_stem_path), sr=None, mono=True)
    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr, units='frames')
    beat_times = librosa.frames_to_time(beat_frames, sr=sr).tolist()
    downbeats = beat_times[::4]  # Estimate: every 4th beat

    beats_result = {
        'bpm': round(float(np.squeeze(tempo)), 2),
        'beats': [round(t, 4) for t in beat_times],
        'downbeats': [round(t, 4) for t in downbeats],
    }
    print(f'BPM: {beats_result["bpm"]}')
    print(f'Beat count: {len(beat_times)}')
else:
    print('No stem available for BPM detection — skipping.')

Using drums stem for BPM detection


## Cell 8 — Key detection

In [ ]:
# Use 'other' stem for key — it has the richest harmonic content.
key_stem_priority = ['other', 'vocals', 'guitar', 'piano', 'bass']
key_stem_path = None
for stem_name in key_stem_priority:
    candidate = stems_source_dir / f'{stem_name}.wav'
    if candidate.exists():
        key_stem_path = candidate
        print(f'Using {stem_name} stem for key detection')
        break

key_result = None

if key_stem_path:
    y, sr = librosa.load(str(key_stem_path), sr=None, mono=True)

    # HPSS: separate harmonic from percussive before chroma analysis.
    # Percussion smears chroma — this improves key detection accuracy.
    y_harmonic, _ = librosa.effects.hpss(y)
    chroma = librosa.feature.chroma_cqt(y=y_harmonic, sr=sr)
    chroma_mean = chroma.mean(axis=1)

    # Krumhansl-Schmuckler key profiles
    major_template = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09,
                                2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
    minor_template = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53,
                                2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
    note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

    best_key, best_mode, best_corr = None, None, -np.inf
    for i in range(12):
        for template, mode in [(major_template, 'major'), (minor_template, 'minor')]:
            corr = np.corrcoef(chroma_mean, np.roll(template, i))[0, 1]
            if corr > best_corr:
                best_corr, best_key, best_mode = corr, i, mode

    root = note_names[best_key]
    key_result = {
        'key': f'{root} {best_mode}',
        'root': root,
        'mode': best_mode,
        'confidence': round(float(best_corr), 3),
    }
    print(f'Key: {key_result["key"]} (confidence: {key_result["confidence"]})')
else:
    print('No stem available for key detection — skipping.')

Using other stem for key detection


## Cell 9 — Note detection (vocals + bass)

Uses `librosa.pyin` — a monophonic pitch tracker. Reliable on isolated
vocals and bass stems. Guitar/piano are polyphonic and are skipped here.

In [ ]:
import json

FREQ_RANGES = {
    'bass':   ('C1', 'G4'),
    'vocals': ('C2', 'C6'),
}
MIN_SEGMENT_DURATION = 0.08
NOTE_CHANGE_THRESHOLD_SEMITONES = 0.5


def extract_note_timeline(audio_path, instrument):
    """
    Identical to backend/note_extraction.py — duplicated here since Colab
    can't import from your local project. If you change the logic in one
    place, update the other to match.
    """
    fmin = librosa.note_to_hz(FREQ_RANGES[instrument][0])
    fmax = librosa.note_to_hz(FREQ_RANGES[instrument][1])

    y, sr = librosa.load(audio_path, sr=None, mono=True)
    f0, voiced_flag, _ = librosa.pyin(y, fmin=fmin, fmax=fmax, sr=sr)
    times = librosa.times_like(f0, sr=sr)

    raw_points = []
    for t, f, voiced in zip(times, f0, voiced_flag):
        if voiced and f is not None and not np.isnan(f):
            raw_points.append((float(t), librosa.hz_to_midi(float(f))))

    if not raw_points:
        return []

    frame_hop = times[1] - times[0] if len(times) > 1 else 0.01
    gap_threshold = frame_hop * 3

    segments = []
    seg_start = raw_points[0][0]
    seg_midi = [raw_points[0][1]]
    prev_t = raw_points[0][0]

    for t, midi in raw_points[1:]:
        avg = sum(seg_midi) / len(seg_midi)
        if abs(midi - avg) <= NOTE_CHANGE_THRESHOLD_SEMITONES and (t - prev_t) <= gap_threshold:
            seg_midi.append(midi)
        else:
            segments.append((seg_start, prev_t, seg_midi))
            seg_start, seg_midi = t, [midi]
        prev_t = t
    segments.append((seg_start, prev_t, seg_midi))

    cleaned = []
    for start, end, midi_values in segments:
        avg_midi = round(sum(midi_values) / len(midi_values))
        note_name = librosa.midi_to_note(avg_midi)
        duration = end - start
        if cleaned:
            ps, pe, pn, pm = cleaned[-1]
            if duration < MIN_SEGMENT_DURATION or (note_name == pn and (start - pe) <= gap_threshold):
                cleaned[-1] = (ps, end, pn, pm)
                continue
        cleaned.append((start, end, note_name, avg_midi))

    return [{'start': round(s, 3), 'end': round(e, 3), 'note': n, 'midi': m} for s, e, n, m in cleaned]


notes_by_stem = {}
for instrument in FREQ_RANGES:
    stem_path = stems_source_dir / f'{instrument}.wav'
    if not stem_path.exists():
        print(f'{instrument}: stem not found, skipping note detection')
        continue
    print(f'Extracting notes for {instrument}...')
    timeline = extract_note_timeline(str(stem_path), instrument)
    notes_by_stem[instrument] = timeline
    print(f'  {len(timeline)} segments found')

print('Note detection complete.')

## Cell 10 — Assemble output & download

Packages everything into a zip. Extract it so `<song_id>/manifest.json`
lands directly inside your local `backend/data/` — not nested inside
an extra folder.

**Data cost:** the zip is mostly the WAV stem files.
Stems are typically 20–60 MB each × 6 stems = 120–360 MB total.
Download this on Wi-Fi.

In [ ]:
import shutil
import json
from pathlib import Path

output_dir = Path('output') / song_id
output_dir.mkdir(parents=True, exist_ok=True)

# Copy stems
stem_files = {}
for wav in stems_source_dir.glob('*.wav'):
    dest = output_dir / wav.name
    shutil.copy(wav, dest)
    stem_files[wav.stem] = str(dest)
    print(f'Copied stem: {wav.stem}')

# Write lyrics
if lyrics_result:
    (output_dir / 'lyrics.json').write_text(json.dumps(lyrics_result))
    print('Wrote lyrics.json')

# Write beats
if beats_result:
    (output_dir / 'beats.json').write_text(json.dumps(beats_result))
    print(f'Wrote beats.json (BPM: {beats_result["bpm"]})')

# Write key
if key_result:
    (output_dir / 'key.json').write_text(json.dumps(key_result))
    print(f'Wrote key.json ({key_result["key"]})')

# Write notes
notes_available = []
for instrument, timeline in notes_by_stem.items():
    (output_dir / f'notes_{instrument}.json').write_text(json.dumps(timeline))
    notes_available.append(instrument)
    print(f'Wrote notes_{instrument}.json ({len(timeline)} segments)')

# Write manifest — this is what the app reads to know what's available
manifest = {
    'song_id': song_id,
    'title': input_path.stem.replace('_', ' '),
    'stems': list(stem_files.keys()),
    'notes_available': notes_available,
    'has_lyrics': lyrics_result is not None,
    'has_beats': beats_result is not None,
    'has_key': key_result is not None,
    'bpm': beats_result['bpm'] if beats_result else None,
    'key': key_result['key'] if key_result else None,
    'demucs_model': DEMUCS_MODEL,
}
(output_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'\nManifest written:')
print(json.dumps(manifest, indent=2))

# ── Save to Google Drive (bypasses browser-download timeout) ──────────────────
# Drive is already mounted at /content/drive from Cell 2.
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/mwtn_outputs')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('\nCreating zip...')
zip_path = shutil.make_archive(song_id, 'zip', root_dir='output', base_dir=song_id)
print(f'Zip created: {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB)')

drive_zip_dest = DRIVE_OUTPUT_DIR / f'{song_id}.zip'
shutil.copy(zip_path, drive_zip_dest)
print(f'\n✓ Saved to Google Drive: My Drive/mwtn_outputs/{song_id}.zip')
print('Open Google Drive in your browser and download the zip at your convenience.')
print(f'Extract so that backend/data/{song_id}/manifest.json exists (no extra nesting).')